# Demo - Sampling Knobs in Text Generation
**Day 3 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/demos/notebooks/demo-sampling-knobs.ipynb)

**Goal:** Show how temperature, top-k, top-p, and beam search change generated
text.

We load a small GPT-2 model and generate from one prompt under several decoding
settings, printing each result so the effect of every knob is visible. Greedy
and low temperature stay repetitive and safe; higher temperature and sampling
add variety; beam search favors fluent, high-probability text.

No API key needed. GPT-2 downloads once (~500 MB), then caches.

In [ ]:
# Scoped install for this demo (safe to re-run; often preinstalled on Colab).
%pip install -q torch transformers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_NAME = "gpt2"
PROMPT = "The best way to learn programming is"
NEW_TOKENS = 30

## 1. Load GPT-2 and a generate helper
`generate` runs one continuation of the prompt under whatever decoding kwargs
we pass, re-seeding first so each run is reproducible.

In [ ]:
def generate(model, tokenizer, label, **kwargs):
    """Generate one continuation of the prompt and print it under a label."""
    set_seed(42)
    inputs = tokenizer(PROMPT, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"\n[{label}]")
    print(f"  {text}")

In [ ]:
print(f"Loading {MODEL_NAME} (downloads once, then cached)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

print(f"\nPrompt: {PROMPT!r}")

## 2. Greedy and temperature
Greedy always picks the top token (repetitive). Low temperature stays focused;
high temperature adds variety and risk.

In [ ]:
generate(model, tokenizer, "greedy (do_sample=False)", do_sample=False)
generate(model, tokenizer, "temperature=0.2 (focused)",
         do_sample=True, temperature=0.2)
generate(model, tokenizer, "temperature=1.2 (wild)",
         do_sample=True, temperature=1.2)

## 3. Top-k, top-p, and beam search
Top-k and top-p (nucleus) restrict sampling to likely tokens. Beam search
explores several paths and favors fluent, high-probability text.

In [ ]:
generate(model, tokenizer, "top_k=10", do_sample=True, top_k=10)
generate(model, tokenizer, "top_p=0.90 (nucleus)",
         do_sample=True, top_p=0.90)
generate(model, tokenizer, "beam search (num_beams=5)",
         do_sample=False, num_beams=5, early_stopping=True)

## Expected output
- Greedy and low temperature produce safe, sometimes repetitive text.
- High temperature and sampling add variety (and occasional nonsense).
- Beam search reads as the most fluent.

**Takeaway:** same prompt, different knobs, different text. That is decoding.